# Bài 36: Đánh giá một bộ phân loại

Mọi bộ phân loại từ đầu đến giờ đều được đánh giá bằng một con số duy nhất: độ chính xác tổng thể. Con số đó che giấu rất nhiều điều. Bài này xây dựng một bộ phân loại 4 lớp trên một tập con **CIFAR-10** mất cân bằng có chủ đích — ảnh chụp thật, không phải hình dạng tổng hợp — và chỉ ra tại sao chỉ riêng độ chính xác có thể gây hiểu lầm, dùng các công cụ tiết lộ điều gì thực sự đang sai: **ma trận nhầm lẫn (confusion matrix)**, và **độ chính xác (precision) và độ phủ (recall) theo từng lớp**.

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Một tập dữ liệu 4 lớp, mất cân bằng

Bốn danh mục ảnh chụp thật từ **CIFAR-10** (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009</a>): mèo, chó, ô tô và xe tải. Tập kiểm tra có đều 150 ảnh mỗi lớp, nhưng tập *huấn luyện* bị cố tình bỏ đói xe tải (90 mẫu, so với 400 mẫu cho mỗi lớp trong ba lớp còn lại) — đại diện cho tình huống thực tế phổ biến khi một số lớp đơn giản là hiếm thu thập hơn các lớp khác. Mèo và chó được đưa vào có chủ đích như một cặp tương tự về thị giác, bất kể mỗi lớp nhận được bao nhiêu dữ liệu huấn luyện.

In [ ]:
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

ensure_cifar10()

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
CLASSES = ['cat', 'dog', 'automobile', 'truck']
CLASS_IDS = {name: CIFAR_LABELS.index(name) for name in CLASSES}

train_imgs, train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    train_imgs.append(imgs); train_labels.append(labels)
train_imgs, train_labels = np.concatenate(train_imgs), np.concatenate(train_labels)
test_imgs, test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

rng = np.random.default_rng(7)

def subset_for_classes(imgs, labels, per_class_counts):
    out_imgs, out_labels = [], []
    for i, name in enumerate(CLASSES):
        idx = np.where(labels == CLASS_IDS[name])[0]
        idx = rng.permutation(idx)[:per_class_counts[name]]
        out_imgs.append(imgs[idx])
        out_labels.append(np.full(len(idx), i, dtype=np.int64))
    out_imgs, out_labels = np.concatenate(out_imgs), np.concatenate(out_labels)
    perm = rng.permutation(len(out_imgs))
    return out_imgs[perm], out_labels[perm]

# training set: cat/dog/automobile well represented, truck deliberately starved
X_train, y_train = subset_for_classes(train_imgs, train_labels,
    {'cat': 400, 'dog': 400, 'automobile': 400, 'truck': 90})
# test set: balanced across all four, so the imbalance is purely a training-time artifact
X_test, y_test = subset_for_classes(test_imgs, test_labels,
    {'cat': 150, 'dog': 150, 'automobile': 150, 'truck': 150})

print('training class counts:', dict(zip(CLASSES, (int(c) for c in np.bincount(y_train, minlength=4)))))
print('test class counts:    ', dict(zip(CLASSES, (int(c) for c in np.bincount(y_test, minlength=4)))))

fig, axes = plt.subplots(1, 4, figsize=(8, 2.2))
for ax, name in zip(axes, CLASSES):
    ax.imshow(X_train[y_train == CLASSES.index(name)][0])
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.show()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc = nn.Linear(64, 4)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))

torch.manual_seed(0)
model = CNN()
opt = torch.optim.Adam(model.parameters(), lr=0.0007)
Xt = torch.tensor(X_train).permute(0, 3, 1, 2); yt = torch.tensor(y_train)
for _ in range(150):
    opt.zero_grad()
    loss = F.cross_entropy(model(Xt), yt)
    loss.backward()
    opt.step()

Xte = torch.tensor(X_test).permute(0, 3, 1, 2)
with torch.no_grad():
    preds = model(Xte).argmax(1).numpy()

acc = (preds == y_test).mean()
print(f'overall test accuracy: {acc:.1%}')

Độ chính xác tổng thể hơn 50% một chút, trên một nhiệm vụ 4 lớp mà mức đoán ngẫu nhiên là 25%, nghe có vẻ là kết quả hợp lý cho một CNN nhỏ huấn luyện từ đầu trên ảnh thật. Nó che giấu một điều quan trọng: mô hình không giỏi như nhau ở cả bốn lớp.

## Ma trận nhầm lẫn

Hàng `i`, cột `j` đếm các mẫu kiểm tra thuộc lớp thực `i` được dự đoán là lớp `j`. Một bộ phân loại hoàn hảo là ma trận chéo; mọi thứ ngoài đường chéo đều là một lỗi cụ thể, gọi tên được.

In [ ]:
cm = np.zeros((4, 4), dtype=int)
for t, p in zip(y_test, preds):
    cm[t, p] += 1

print('confusion matrix (rows=true, cols=predicted):')
print(f'{"":>12}' + ''.join(f'{s:>12}' for s in CLASSES))
for i, s in enumerate(CLASSES):
    print(f'{s:>12}' + ''.join(f'{cm[i, j]:>12}' for j in range(4)))

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES, rotation=45)
ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.title('Confusion matrix')
plt.tight_layout()
plt.show()

Ảnh động bên dưới điền vào cùng ma trận đó từng ô một: xanh lục cho trúng (trên đường chéo), đỏ cho trượt (ngoài đường chéo).

<img src="../animations/confusion_matrix_demo.gif" alt="Ảnh động ma trận nhầm lẫn được điền vào từng ô, từng hàng, với các ô trúng trên đường chéo tô xanh lục và các ô trượt ngoài đường chéo tô đỏ, kết thúc với bản đồ nhiệt đầy đủ" style="display: block; margin: 0 auto; max-width: 620px;">

## Dương tính thật, dương tính giả, âm tính thật, âm tính giả

Mỗi ô của ma trận nhầm lẫn ở trên là một loại đúng hoặc sai cụ thể, nhưng từ vựng tiêu chuẩn để nói về chúng là nhị phân: chọn một lớp và chỉ hỏi "có phải lớp này hay không?" Thu gọn ma trận 4 lớp thành "xe tải so với mọi thứ khác" cho ra đúng bốn kết cục:

- **Dương tính thật (true positive — TP)**: thực sự là xe tải, dự đoán là xe tải. Trúng.
- **Âm tính giả (false negative — FN)**: thực sự là xe tải, dự đoán là thứ khác. Trượt — mô hình để nó lọt qua.
- **Dương tính giả (false positive — FP)**: thực sự *không phải* xe tải, nhưng vẫn dự đoán là xe tải. Báo động nhầm.
- **Âm tính thật (true negative — TN)**: thực sự không phải xe tải, dự đoán đúng là không-phải-xe-tải.

Đây là cùng bảng 2x2 nằm bên dưới việc đánh giá mọi bộ phân loại nhị phân (kết quả "dương tính/âm tính" của một xét nghiệm y tế, quyết định "thư rác/không phải thư rác" của một bộ lọc thư rác) — một ma trận nhầm lẫn nhiều lớp chỉ là bảng này được tính một lần cho mỗi lớp, với mọi thứ ngoài hàng/cột của lớp đó được gộp thành "không phải lớp này".

In [ ]:
cls = CLASSES.index('truck')
tp = cm[cls, cls]
fn = cm[cls, :].sum() - tp          # true truck, predicted something else
fp = cm[:, cls].sum() - tp          # predicted truck, actually something else
tn = cm.sum() - tp - fn - fp        # everything else, correctly not called truck

print(f'{"":>18}{"predicted truck":>20}{"predicted NOT truck":>24}')
print(f'{"actually truck":>18}{tp:>20}{fn:>24}')
print(f'{"actually NOT truck":>18}{fp:>20}{tn:>24}')
print()
print(f'TP={tp}, FP={fp}, FN={fn}, TN={tn}, total={tp+fp+fn+tn} (test set size={len(y_test)})')

Ảnh động bên dưới luân phiên qua cả bốn lớp, thu gọn ma trận đầy đủ thành bảng TP/FP/FN/TN của mỗi lớp &mdash; cùng phép thu gọn đã thực hiện ở trên cho 'xe tải', được tổng quát hóa.

<img src="../animations/tpfpfn_demo.gif" alt="Ảnh động luân phiên qua từng lớp trong bốn lớp, thu gọn ma trận nhầm lẫn đầy đủ thành bảng 2x2 TP, FP, FN, TN của lớp đó" style="display: block; margin: 0 auto; max-width: 800px;">

## Độ chính xác và độ phủ

Hai con số cho mỗi lớp, tính trực tiếp từ TP/FP/FN:
- **Độ phủ (recall)** = TP / (TP + FN) — trong tất cả những gì thực sự *là* lớp `i`, mô hình bắt được bao nhiêu phần? Độ phủ thấp nghĩa là mô hình thường bỏ sót lớp đó.
- **Độ chính xác (precision)** = TP / (TP + FP) — trong tất cả những gì mô hình *gọi* là lớp `i`, bao nhiêu phần thực sự đúng? Độ chính xác thấp nghĩa là mô hình thường "kêu sói đến" với lớp đó.

(Một cặp liên quan nhưng ít cần đến hơn, xây dựng từ hai góc phần tư còn lại: **độ đặc hiệu (specificity)** = TN / (TN + FP), mô hình tránh báo động nhầm trên lớp âm tốt đến đâu, và phần bù của nó là **tỉ lệ dương tính giả (false positive rate)** = FP / (FP + TN) = 1 − độ đặc hiệu.)

In [ ]:
print(f'{"class":>10} {"precision":>10} {"recall":>8} {"support":>8}')
for i, name in enumerate(CLASSES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    recall = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    support = cm[i, :].sum()
    print(f'{name:>10} {precision:>10.2f} {recall:>8.2f} {support:>8}')

Ảnh động bên dưới luân phiên qua từng lớp, cho thấy độ chính xác và độ phủ dưới dạng tỉ lệ của cùng các số đếm TP/FP/FN: một thanh độ phủ chia thành bắt được (TP) và bỏ sót (FN), một thanh độ chính xác chia thành đúng (TP) và báo động nhầm (FP).

<img src="../animations/precision_recall_demo.gif" alt="Ảnh động luân phiên qua từng lớp trong bốn lớp, cho thấy độ phủ dưới dạng một thanh chia thành dương tính thật và âm tính giả, và độ chính xác dưới dạng một thanh chia thành dương tính thật và dương tính giả, với tỉ lệ thu được được ghi nhãn cho mỗi thanh" style="display: block; margin: 0 auto; max-width: 820px;">

Để tra cứu nhanh, vì các thuật ngữ này có nhiều tên gọi khác nhau trong các lĩnh vực:

| thuật ngữ | công thức | còn gọi là |
|---|---|---|
| Precision (độ chính xác) | TP / (TP + FP) | Giá trị tiên đoán dương (Positive Predictive Value — PPV) |
| Recall (độ phủ) | TP / (TP + FN) | Độ nhạy (Sensitivity), Tỉ lệ dương tính thật (True Positive Rate — TPR) |
| Specificity (độ đặc hiệu) | TN / (TN + FP) | Tỉ lệ âm tính thật (True Negative Rate — TNR) |
| False Positive Rate (tỉ lệ dương tính giả) | FP / (TN + FP) = 1 − Specificity | Fallout, Tỉ lệ báo động nhầm |
| False Negative Rate (tỉ lệ âm tính giả) | FN / (TP + FN) = 1 − Recall | Tỉ lệ bỏ sót (Miss Rate) |

### Quy tắc Bayes

Để thấy tại sao độ chính xác lại hành xử như vậy, việc nhận ra một mối liên hệ sâu hơn có thể giúp ích. **Quy tắc Bayes (Bayes' rule)** liên hệ một xác suất có điều kiện với chiều ngược lại của nó: $P(A\mid B) = P(B\mid A)\,P(A)\,/\,P(B)$. Quy tắc Bayes nối độ chính xác, $P(\text{actually } i\mid\text{predicted } i)$, với độ nhạy/độ đặc hiệu, $P(\text{predicted } i\mid\text{actually } i)$:

$$P(\text{actually } i\mid\text{predicted } i) = \frac{P(\text{predicted } i\mid\text{actually } i)\,P(\text{actually } i)}{P(\text{predicted } i)}.$$

Số hạng $P(\text{actually } i)$ — **tỉ lệ cơ sở (base rate)** hay **tiên nghiệm (prior)**, lớp `i` phổ biến đến đâu ngay từ đầu — quan trọng không kém việc bộ phân loại tốt đến đâu.

Ví dụ minh họa kinh điển: một căn bệnh có tỉ lệ mắc thực là 1%, và một xét nghiệm "chính xác 99%" (độ nhạy 99%, độ đặc hiệu 99%). Bạn có kết quả dương tính — khả năng bạn thực sự mắc bệnh là bao nhiêu? Trực giác nói ≈99%, nhưng đáp án thực gần với 50% hơn. Lý do: trong 10.000 người, 100 người thực sự mắc bệnh, và xét nghiệm bắt được 99 người trong số đó (TP=99). Trong 9.900 người khỏe mạnh, tỉ lệ dương tính giả 1% đánh dấu thêm khoảng 99 người (FP=99) — nên trong số tất cả những người có kết quả dương tính, chỉ khoảng một nửa thực sự mắc bệnh: độ chính xác = 99/198 ≈ 50%, còn lâu mới đạt 99%. Một tỉ lệ cơ sở thấp khiến các dương tính giả từ đa số khỏe mạnh khổng lồ lấn át các dương tính thật từ thiểu số hiếm hoi, nhỏ bé.

Xe tải cho thấy cùng hiệu ứng đó: nó là lớp hiếm nhất ở đây, và ngay cả độ chính xác 72% khá đáng nể của nó (bảng ở trên) cũng đã phải chống chọi với cùng phép tính bất lợi khiến độ chính xác (còn gọi là "giá trị tiên đoán dương") trở nên khắc nghiệt với bất kỳ tình trạng hiếm nào, y tế hay không.

### Đường cong độ chính xác-độ phủ

Từ đầu đến giờ, độ chính xác và độ phủ của mỗi lớp đến từ một quy tắc quyết định duy nhất, cố định: lớp nào có điểm số cao nhất thì thắng (argmax cứng). Nhưng mô hình thực ra không đưa ra một phán quyết duy nhất &mdash; với "xe tải so với mọi thứ khác", nó đưa ra một điểm số liên tục (xác suất softmax gán cho xe tải) cho mỗi ảnh kiểm tra. Phân ngưỡng trực tiếp điểm số đó và quét ngưỡng từ 0 đến 1 vẽ ra một **đường cong độ chính xác-độ phủ (precision-recall curve)**: sự đánh đổi đầy đủ giữa bắt được nhiều xe tải hơn và kêu sói đến ít hơn. Argmax cứng không hoàn toàn là cùng một quy tắc quyết định &mdash; nó phụ thuộc vào cách điểm số xe tải so sánh với *ba lớp còn lại*, chứ không chỉ việc nó có vượt qua một con số cố định nào đó không &mdash; nên điểm mà nó thực sự rơi vào (đánh dấu màu đỏ) nằm gần, nhưng không chính xác trên, đường cong một-điểm-số này.

Đường cong thô bên dưới lởm chởm: với quá ít mẫu xe tải, một dương tính giả không may (hoặc may mắn) duy nhất có thể làm độ chính xác tụt xuống rồi hồi phục khi ngưỡng được nới lỏng thêm, đó không phải là thay đổi thực sự trong sự đánh đổi bên dưới &mdash; chỉ là nhiễu từ một tập kiểm tra nhỏ. Cách khắc phục tiêu chuẩn (được dùng, chẳng hạn, để tính "độ chính xác trung bình — average precision") là thay độ chính xác của mỗi điểm bằng độ chính xác *tốt nhất* đạt được ở độ phủ đó hoặc cao hơn, tạo ra một đường bao không tăng đơn điệu "đổ bóng" sang trái lên các chỗ tụt nhiễu.

In [ ]:
with torch.no_grad():
    probs = F.softmax(model(Xte), dim=1).numpy()

scores = probs[:, cls]          # softmax probability assigned to 'truck'
is_truck = (y_test == cls)

thresholds = np.linspace(0, 1, 101)
precisions, recalls = [], []
for t in thresholds:
    pred_pos = scores >= t
    tp_t = (pred_pos & is_truck).sum()
    fp_t = (pred_pos & ~is_truck).sum()
    fn_t = (~pred_pos & is_truck).sum()
    precisions.append(tp_t / (tp_t + fp_t) if tp_t + fp_t > 0 else 1.0)
    recalls.append(tp_t / (tp_t + fn_t) if tp_t + fn_t > 0 else 0.0)

# the raw curve is jagged (precision isn't monotonic as the threshold relaxes --
# one unlucky false positive can make it dip, then recover). The standard fix,
# used e.g. to compute "average precision": replace each point's precision with
# the BEST precision achieved at that recall or higher -- a monotonically
# non-increasing envelope that "casts a shadow" leftward over the noisy dips.
# Different thresholds can land on the exact same recall (relaxing the
# threshold past a non-truck image changes precision but not recall), so ties
# are first collapsed to their best precision before taking the running max --
# otherwise a later, worse-precision point at the same recall can silently
# undercut an earlier, better one.
order = np.argsort(recalls)
r_sorted = np.array(recalls)[order]
p_sorted = np.array(precisions)[order]
uniq_r, inverse = np.unique(r_sorted, return_inverse=True)
uniq_p = np.zeros_like(uniq_r)
np.maximum.at(uniq_p, inverse, p_sorted)
envelope = np.maximum.accumulate(uniq_p[::-1])[::-1]

plt.plot(recalls, precisions, marker='.', color='#aac4e0', linewidth=1.2, label='raw (noisy)')
# where='pre': each point's value extends LEFTWARD (toward lower recall) onto
# the gap before it -- matching max_{r'>=r} P(r'), and leaving points that are
# already their own local max untouched, instead of dragging a peak forward
# onto later points that don't need it.
plt.step(uniq_r, envelope, where='pre', color='#1f6fb4', linewidth=2.2,
          label='envelope (monotonic)')
plt.scatter([tp / (tp + fn)], [tp / (tp + fp)], color='red', marker='x', s=60, zorder=5,
            label='hard-argmax point')
plt.xlabel('recall'); plt.ylabel('precision')
plt.title('truck: precision-recall curve as the decision threshold sweeps')
plt.legend(fontsize=8)
plt.show()

Ảnh động bên dưới làm cụ thể cơ chế đằng sau đường cong đó: một ngưỡng quét qua điểm số xe tải của mọi ảnh kiểm tra (sắp xếp từ trái sang phải), tô màu mỗi ảnh là TP/FP/FN/TN trực tiếp khi nó vượt qua, trong khi điểm (độ phủ, độ chính xác) bò dọc theo cùng đường cong hiển thị ở trên.

<img src="../animations/pr_sweep_demo.gif" alt="Ảnh động quét ngưỡng độ chính xác-độ phủ: một ngưỡng quyết định trượt qua các ảnh kiểm tra được sắp xếp theo điểm số xe tải softmax của chúng, tô màu mỗi ảnh là dương tính thật, dương tính giả, âm tính giả hoặc âm tính thật khi nó vượt qua, trong khi một điểm vẽ ra đường cong độ chính xác-độ phủ thu được trên một khung thứ hai" style="display: block; margin: 0 auto; max-width: 900px;">

Xe tải — lớp bị bỏ đói chỉ còn 90 ảnh huấn luyện, so với 400 cho mỗi lớp khác — có độ chính xác cao nhưng độ phủ thấp: khi mô hình nói "xe tải", nó thường đúng, nhưng nó không nhận ra được tuyệt đại đa số xe tải thực sự, thay vào đó mặc định chọn những lớp mà nó đã thấy nhiều trong khi huấn luyện (chủ yếu là "ô tô", lớp được đại diện tốt gần nhất về thị giác). Đó là dấu hiệu tiêu chuẩn của mất cân bằng lớp, và nó hoàn toàn vô hình trong con số độ chính xác tổng thể duy nhất trước đó. Ngược lại, mèo và chó đều được đại diện tốt trong huấn luyện nhưng bị nhầm *với nhau* nhiều hơn hẳn so với với ô tô hay xe tải — một kiểu thất bại hoàn toàn khác, gây ra bởi sự tương đồng thị giác thực sự giữa hai loài vật chứ không phải do thiếu dữ liệu. Ô tô kết thúc với độ phủ cao bất thường nhưng độ chính xác không mấy tốt, cùng lý do khiến độ phủ của xe tải bị ảnh hưởng: nó hấp thụ các lỗi phân loại của lớp xe tải bị bỏ đói, vì "xe có bánh" là một lựa chọn dự phòng dễ đoán khi mô hình không phân biệt được xe tải với ô tô.

Bài học thực tế: luôn kiểm tra ma trận nhầm lẫn và các thước đo theo từng lớp trước khi tin vào một con số độ chính xác duy nhất, đặc biệt trên một tập dữ liệu mà các lớp không cân bằng.

### Bài tập

1. Tăng số ảnh xe tải huấn luyện từ 90 lên 400 (khớp với ba lớp còn lại) và chạy lại. Độ phủ của xe tải có hồi phục không? Sự nhầm lẫn mèo/chó có cũng cải thiện, hay vẫn tồn tại — và tại sao nhiều dữ liệu hơn lại không khắc phục được một vấn đề gây ra bởi sự tương đồng thị giác thực sự thay vì sự khan hiếm dữ liệu?
2. Thử đánh trọng số hàm mất mát theo nghịch đảo tần suất lớp (`F.cross_entropy(logits, yt, weight=class_weights)`, trong đó `class_weights[i] = 1 / count(class i)`) thay vì thu thập thêm ảnh xe tải. Nó có lấy lại được độ phủ của xe tải không, và với cái giá nào đối với độ chính xác của các lớp khác?
3. Tính **điểm F1 (F1 score)** (trung bình điều hòa của độ chính xác và độ phủ, `2 * p * r / (p + r)`) cho mỗi lớp. Tại sao F1 có thể là một con số duy nhất tốt hơn để theo dõi theo từng lớp so với độ chính xác tổng thể, khi độ chính xác tổng thể chỉ có ý nghĩa khi gộp chung?